In [ ]:
import os, sys, json
import pandas as pd
import numpy as np

sys.path.append("..")

from matplotlib import pyplot as plt

plt.rcParams.update({
    "font.size": 12,
    "text.usetex": True,
    "font.family": "serif",
    "text.latex.preamble": r"\usepackage{amsmath}",
    "font.serif": ["Roman"],
})

if plt.rcParams["text.usetex"]:
    print("LaTeX is enabled for text rendering in matplotlib. Disable if not Latex is installed or if you encounter issues.")
else:
    print("LaTeX is disabled for text rendering in matplotlib. Enable if you have LaTeX installed and want to use it for rendering text.")

In [ ]:
BO_probes_path = os.path.join("Results", "BO_PEresults_BO_20260618_113945", "probes.csv")
print(f"Loading BO probes from {BO_probes_path}...")
if not os.path.exists(BO_probes_path):
    raise FileNotFoundError(f"BO probes file not found at {BO_probes_path}. Please enter a valid path.")


BO_probes = pd.read_csv(BO_probes_path, index_col=0)
BO_probes.head()

In [ ]:
BO_probes.drop(np.where(BO_probes["wls"] > 1e6)[0], inplace=True)

np.where(BO_probes["wls"].min() == BO_probes["wls"])[0]
best_probe = BO_probes.iloc[np.where(BO_probes["wls"].min() == BO_probes["wls"])[0][0]]
print(f"Best probe (iteration {best_probe.name - 81}):")
print(best_probe)

In [ ]:
best_so_far = BO_probes["wls"].expanding().min()
best_so_far

In [ ]:
y_min = 200
y_max = None

ncols = 1
nrows = 1
scaling_factor = 1.0

figsize = (ncols*5*scaling_factor, nrows*3*scaling_factor)

fig, ax = plt.subplots(ncols=ncols, nrows=nrows, figsize=figsize, sharey=True, constrained_layout=True)

ax.plot(BO_probes.index, BO_probes["wls"], "o", markersize=2, label = "Samples")
ax.plot(BO_probes.index, BO_probes["wls"].expanding().min(), "-", label="Best so far")
ax.plot(best_probe.name, best_probe["wls"], "rx", markersize=10, label = "Best overall")
ax.axvline(x = 80.5, color="black", linestyle="dashed", label="End of initial design")
ax.set_yscale("log")
# ax.set_ylim([y_min, y_max])
ax.set_xlim([BO_probes.index.min(), BO_probes.index.max()])
ax.set_xlabel("BO iteration $k$")
ax.set_ylabel(r"$\mathrm{WLS} \, / \, -$")

ax.legend(loc = "upper right")

os.makedirs(os.path.join(os.path.dirname(BO_probes_path), "figs"), exist_ok=True)
fig.savefig(os.path.join(os.path.dirname(BO_probes_path), "figs", "BO_convergence_2.png"), dpi=1200)
print(f"Saved BO convergence figure to {os.path.join(os.path.dirname(BO_probes_path), 'figs', 'BO_convergence_2.png')}")
fig.savefig(os.path.join(os.path.dirname(BO_probes_path), "figs", "BO_convergence_2.pdf"), dpi=1200)
print(f"Saved BO convergence figure to {os.path.join(os.path.dirname(BO_probes_path), 'figs', 'BO_convergence_2.pdf')}")

# Simulate

In [ ]:
from parameter_estimation import load_experiment, simulate_experiment, _recalc_exp_states


cwd = os.getcwd()
best_trajectory_name = "260414_1"
best_trajectory_path = os.path.join(
    os.getcwd(),
    "..",
    "data",
    "experimentaldata",
    best_trajectory_name,
    "results",
    "trajectories.pkl"
)

best_experiment = load_experiment(best_trajectory_path)


worst_trajectory_name = "260312"
worst_trajectory_path = os.path.join(
    os.getcwd(),
    "..",
    "data",
    "experimentaldata",
    worst_trajectory_name,
    "results",
    "trajectories.pkl"
)

worst_experiment = load_experiment(worst_trajectory_path)

In [ ]:
with open(os.path.join(os.path.dirname(BO_probes_path), "best.json"), "r") as f:
    best_probe = json.load(f)
best_probe

y_sim_best = simulate_experiment(best_probe["E_murphree"], best_probe["rr_err"], best_probe["x_N2"], best_probe["h_weir"], best_experiment)
if y_sim_best is None:
    raise RuntimeError(f"Simulation failed for experiment {best_experiment.name}")
exp_states_best = _recalc_exp_states(best_experiment, best_probe["x_N2"])

y_sim_worst = simulate_experiment(best_probe["E_murphree"], best_probe["rr_err"], best_probe["x_N2"], best_probe["h_weir"], worst_experiment)
if y_sim_worst is None:
    raise RuntimeError(f"Simulation failed for experiment {worst_experiment.name}")
exp_states_worst = _recalc_exp_states(worst_experiment, best_probe["x_N2"])

In [ ]:
alpha = 0.2
percentage_of_ylim = 0.2

ncols = 2
nrow = 3
scaling_factor = 0.6
figsize = (ncols * 5 * scaling_factor, nrow * 3.5 * scaling_factor)

lbx = np.array([30.0, 0.0, 0.85, 0.0, 360.0])
ubx = np.array([56.0, 15e3, 0.8955, 0.2286, 375.5])
# Add the backoff
# lbx[2] += 0.005
ubx[1] = 6e3

lbu = np.array([0.7, 3.0])
ubu = np.array([1.0, 5.0])



plt.close("all")

fig, ax = plt.subplots(ncols=ncols, nrows=nrow, figsize=figsize, constrained_layout=True)

y_data_experiment = exp_states_best
u_data_experiment = best_experiment.U
time_data_experiment = best_experiment.time

time_data_experiment = time_data_experiment / 60.0


# Plot the experiments
label = "Experiment"
ax[0, 0].plot(time_data_experiment, y_data_experiment[:, 1] * 1e-3, marker = "x", linestyle = "None", label = label)
ax[1, 0].plot(time_data_experiment, y_data_experiment[:, 2], marker = "x", linestyle = "None")
ax[2, 0].plot(time_data_experiment, y_data_experiment[:, 4] - 273.15, marker = "x", linestyle = "None")


t_min_best = time_data_experiment[0]
t_max_best = time_data_experiment[-1]


# Plot the simulation with the best probe parameters
y_data_simulation = y_sim_best
u_data_simulation = best_experiment.U
time_data_simulation = best_experiment.time

time_data_simulation = time_data_simulation / 60.0

label = f"Simulation"
ax[0, 0].plot(time_data_simulation, y_data_simulation[:, 1] * 1e-3, label = label, linestyle = 'solid')
ax[1, 0].plot(time_data_simulation, y_data_simulation[:, 2], linestyle = 'solid')
ax[2, 0].plot(time_data_simulation, y_data_simulation[:, 4] - 273.15, linestyle = 'solid')



ax[0, 0].set_ylabel(r"$m_\mathrm{P}~/~$kg")
ax[0, 0].fill_between(np.array([t_min_best, t_max_best]).flatten(), lbx[1] * 1e-3, lbx[1] * 1e-3 - (ubx[1] * 1e-3 - lbx[1] * 1e-3), color="tab:red", alpha=alpha, label = r"Constraints")
ax[0, 0].set_ylim([lbx[1] * 1e-3 - percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3), ubx[1] * 1e-3 + percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3)])
ax[0, 0].set_yticks(np.arange(0.0, 6.1, step=2.0))


ax[1, 0].set_ylabel(r"$w_\mathrm{P}~/~-$")
ax[1, 0].set_yticks(np.arange(0.81, 0.931, step=0.02))

ax[2, 0].set_ylabel(r"$T_\mathrm{B}~/~^{\circ}$C")
ax[2, 0].fill_between(np.array([t_min_best, t_max_best]).flatten(), lbx[4] - 273.15, lbx[4] - 273.15 - (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
ax[2, 0].fill_between(np.array([t_min_best, t_max_best]).flatten(), 100.0, 100.0 + (100.0 + 273.15 - lbx[4]), color="tab:red", alpha=alpha)
ax[2, 0].set_ylim([lbx[4] - 273.15 - percentage_of_ylim * (100.0 + 273.15 - lbx[4]), 100.0 + percentage_of_ylim * (100.0 + 273.15 - lbx[4])])
ax[2, 0].set_yticks(np.arange(85, 100.1, step=5.0))

ax[0, 0].set_title(r"\textbf{Best}")
ax[-1, 0].set_xlabel(r"Time $t~/~$min")

ax[0, 0].set_xlim([t_min_best, t_max_best])
ax[1, 0].set_xlim([t_min_best, t_max_best])
ax[2, 0].set_xlim([t_min_best, t_max_best])

ax[0, 0].set_xticks(np.arange(0, t_max_best+1, step=15))
ax[1, 0].set_xticks(np.arange(0, t_max_best+1, step=15))
ax[2, 0].set_xticks(np.arange(0, t_max_best+1, step=15))


# Worst performance
y_data_experiment = exp_states_worst
u_data_experiment = worst_experiment.U
time_data_experiment = worst_experiment.time

time_data_experiment = time_data_experiment / 60.0


# Plot the experiments
label = "Experiment"
ax[0, 1].plot(time_data_experiment, y_data_experiment[:, 1] * 1e-3, marker = "x", linestyle = "None")
ax[1, 1].plot(time_data_experiment, y_data_experiment[:, 2], marker = "x", linestyle = "None")
ax[2, 1].plot(time_data_experiment, y_data_experiment[:, 4] - 273.15, marker = "x", linestyle = "None")


t_min_worst = time_data_experiment[0]
t_max_worst = time_data_experiment[-1]


# Plot the simulation with the worst probe parameters
y_data_simulation = y_sim_worst
u_data_simulation = worst_experiment.U
time_data_simulation = worst_experiment.time

time_data_simulation = time_data_simulation / 60.0

label = f"Simulation"
ax[0, 1].plot(time_data_simulation, y_data_simulation[:, 1] * 1e-3, linestyle = 'solid')
ax[1, 1].plot(time_data_simulation, y_data_simulation[:, 2], linestyle = 'solid')
ax[2, 1].plot(time_data_simulation, y_data_simulation[:, 4] - 273.15, linestyle = 'solid')



ax[0, 1].set_ylabel(r"$m_\mathrm{P}~/~$kg")
ax[0, 1].fill_between(np.array([t_min_worst, t_max_worst]).flatten(), lbx[1] * 1e-3, lbx[1] * 1e-3 - (ubx[1] * 1e-3 - lbx[1] * 1e-3), color="tab:red", alpha=alpha)
ax[0, 1].set_ylim([lbx[1] * 1e-3 - percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3), ubx[1] * 1e-3 + percentage_of_ylim * (ubx[1] * 1e-3 - lbx[1] * 1e-3)])
ax[0, 1].set_yticks(np.arange(0.0, 6.1, step=2.0))


ax[1, 1].set_ylabel(r"$w_\mathrm{P}~/~-$")
ax[1, 1].set_yticks(np.arange(0.81, 0.931, step=0.02))

ax[2, 1].set_ylabel(r"$T_\mathrm{B}~/~^{\circ}$C")
ax[2, 1].fill_between(np.array([t_min_worst, t_max_worst]).flatten(), lbx[4] - 273.15, lbx[4] - 273.15 - (ubx[4] - lbx[4]), color="tab:red", alpha=alpha)
ax[2, 1].fill_between(np.array([t_min_worst, t_max_worst]).flatten(), 100.0, 100.0 + (100.0 + 273.15 - lbx[4]), color="tab:red", alpha=alpha)
ax[2, 1].set_ylim([lbx[4] - 273.15 - percentage_of_ylim * (100.0 + 273.15 - lbx[4]), 100.0 + percentage_of_ylim * (100.0 + 273.15 - lbx[4])])
ax[2, 1].set_yticks(np.arange(85, 100.1, step=5.0))

ax[0, 1].set_title(r"\textbf{Worst}")
ax[-1, 1].set_xlabel(r"Time $t~/~$min")

ax[0, 1].set_xlim([t_min_worst, t_max_worst])
ax[1, 1].set_xlim([t_min_worst, t_max_worst])
ax[2, 1].set_xlim([t_min_worst, t_max_worst])

ax[0, 1].set_xticks(np.arange(0, t_max_worst+1, step=15))
ax[1, 1].set_xticks(np.arange(0, t_max_worst+1, step=15))
ax[2, 1].set_xticks(np.arange(0, t_max_worst+1, step=15))

for axis in ax.flatten():
    axis.grid(True)

fig.legend(loc = "outside lower center", ncol=3)

figpath = os.path.join(os.path.dirname(BO_probes_path), "figs", "trajectories")
os.makedirs(figpath, exist_ok=True)
fig.savefig(os.path.join(figpath, f"fit_best_vs_worst_present_v1.png"), dpi=1200)
print(f"Saved best vs worst trajectory figure to {os.path.join(figpath, f'fit_best_vs_worst_present_v1.png')}")
fig.savefig(os.path.join(figpath, f"fit_best_vs_worst_present_v1.pdf"), dpi=300)
print(f"Saved best vs worst trajectory figure to {os.path.join(figpath, f'fit_best_vs_worst_present_v1.pdf')}")